
# Data Prep & Transformation · Aula 4
## Limpeza e padronização: do bronze ao silver
**Luciano Gillieron Gavinho, PhD.** · MBA em Engenharia de Dados · Mackenzie

Os CSVs da Olist, do jeito que chegam, são a nossa camada **bronze**: o dado bruto, que não se altera. Hoje construímos a camada **silver**: o mesmo dado, com tipos corretos, textos padronizados, duplicatas tratadas e valores fora do padrão marcados. Cada regra aplicada fica **registrada** e cada resultado fica **testado**.

**Como a aula funciona**
- **Parte 1** (depois do primeiro bloco de slides): aquecimento A e exercícios 1 e 2. Medir antes de limpar.
- **Parte 2** (depois do intervalo): exercícios 3 a 7. Corrigir, padronizar, deduplicar, marcar e montar a silver.



### Termos usados hoje
| Termo | Significado |
|---|---|
| **Bronze / Silver / Gold** | Camadas da arquitetura vista na Aula 1. Bronze: dado bruto, como chegou. Silver: dado limpo, tipado e padronizado. Gold: dado pronto para consumo (estrela, wide table). |
| **Perfil dos dados** (*data profiling*) | Medição sistemática de cada coluna antes de qualquer limpeza: tipo, nulos, mínimo, máximo, valores distintos. |
| `SUMMARIZE` | Comando do DuckDB que devolve o perfil de uma tabela, uma linha por coluna. |
| **Nulo** (`NULL`) | Ausência de valor. Pode ser esperada (o pedido ainda não foi entregue) ou uma falha (o pedido foi entregue e a data não foi gravada). |
| `TRY_CAST(x AS tipo)` | Tenta converter `x` para o tipo pedido; se não conseguir, devolve nulo em vez de interromper a consulta. |
| `lpad(texto, n, '0')` | Completa o texto com zeros à esquerda até ter `n` caracteres. |
| `strip_accents`, `lower`, `trim` | Removem acentos, convertem para minúsculas e tiram espaços das pontas. |
| `regexp_replace` | Substitui o trecho de texto que casa com uma **expressão regular** (padrão de busca em texto). |
| **Duplicata exata** | Duas linhas idênticas em todas as colunas. |
| **Duplicata lógica** | Duas linhas que representam a mesma coisa, mas diferem em alguma coluna. |
| `QUALIFY` | Filtro do DuckDB aplicado depois de uma função de janela, como `ROW_NUMBER()`. Serve para ficar com uma linha por grupo. |
| **IQR** | *Interquartile range*, intervalo interquartil: Q3 − Q1, a distância entre o 1º e o 3º quartil. Base de uma regra comum para valores extremos. |
| **Flag** | Coluna de marcação (verdadeiro ou falso) que sinaliza um problema sem apagar a linha. |
| `jaro_winkler_similarity` | Mede a semelhança entre dois textos, de 0 (nada parecidos) a 1 (iguais). Ajuda a sugerir correções de digitação. |



## 0 · Preparação do ambiente
Execute as duas células abaixo. A primeira baixa os dados do repositório da Olist com `git clone` e instala o DuckDB com `pip`. A segunda registra cada CSV como uma *view* (consulta salva com nome): essas views são a nossa camada **bronze**. Ela também cria três funções:
- `sql()` executa uma consulta e devolve uma tabela do pandas;
- `registrar()` anota cada regra de limpeza aplicada, com o número de linhas afetadas;
- `testar()` executa uma consulta que **deve devolver zero linhas** e mostra se o teste passou ou falhou.


In [1]:

import os, subprocess
if not os.path.exists("work-at-olist-data"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/olist/work-at-olist-data.git"], check=True)
subprocess.run(["pip", "install", "-q", "duckdb"], check=False)
print(sorted(os.listdir("work-at-olist-data/datasets")))


['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [2]:

import duckdb, pandas as pd
DATA = "work-at-olist-data/datasets"
con = duckdb.connect()

FONTES = {
    "customers":  "olist_customers_dataset.csv",
    "geolocation":"olist_geolocation_dataset.csv",
    "orders":     "olist_orders_dataset.csv",
    "items":      "olist_order_items_dataset.csv",
    "payments":   "olist_order_payments_dataset.csv",
    "reviews":    "olist_order_reviews_dataset.csv",
    "products":   "olist_products_dataset.csv",
    "sellers":    "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
for nome, arq in FONTES.items():
    con.execute(f"CREATE OR REPLACE VIEW {nome} AS SELECT * FROM read_csv_auto('{DATA}/{arq}', header=true)")

def sql(q):
    """Executa uma consulta no DuckDB e devolve um DataFrame."""
    return con.sql(q).df()

REGRAS = []
def registrar(tabela, regra, acao, linhas):
    """Anota uma regra de limpeza. acao: 'corrige', 'marca' ou 'remove'."""
    REGRAS.append({"tabela": tabela, "regra": regra, "acao": acao, "linhas_afetadas": int(linhas)})

def testar(nome, consulta):
    """A consulta deve devolver as linhas que VIOLAM a regra. Zero linhas = teste passou."""
    n = con.sql(f"SELECT COUNT(*) FROM ({consulta})").fetchall()[0][0]
    print(("PASSOU" if n == 0 else "FALHOU"), "|", nome, "|", n, "linha(s) violando a regra")
    return n

sql("SELECT COUNT(*) AS pedidos FROM orders")


,pedidos
0,99441



## A · Aquecimento: exemplos resolvidos
Cinco exemplos prontos com a sintaxe de hoje. Eles usam a tabela `payments` (pagamentos) ou pequenas tabelas escritas à mão com `VALUES`, e **não** respondem aos exercícios.

### A.1 · Perfil de uma tabela com `SUMMARIZE`
Uma linha por coluna: tipo, mínimo, máximo, valores distintos aproximados (`approx_unique`), percentual de nulos (`null_percentage`) e total de linhas (`count`).


In [3]:

sql("""
SELECT column_name, column_type, min, max, approx_unique, null_percentage, count
FROM (SUMMARIZE payments)
""")


,column_name,column_type,min,max,approx_unique,null_percentage,count
0,order_id,VARCHAR,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244,87618,0.0,103886
1,payment_sequential,BIGINT,1,29,27,0.0,103886
2,payment_type,VARCHAR,boleto,voucher,5,0.0,103886
3,payment_installments,BIGINT,0,24,25,0.0,103886
4,payment_value,DOUBLE,0.0,13664.08,31557,0.0,103886



**Leitura:** `payment_installments` (parcelas) tem mínimo **0**, o que não faz sentido para um pagamento; `payment_type` tem 5 valores, e um deles é `not_defined`. Duas pistas para investigar, achadas sem escrever nenhuma consulta específica. Atenção: `approx_unique` é uma **estimativa**, não uma contagem exata.



### A.2 · Tipos: converter sem quebrar
`CAST` interrompe a consulta quando a conversão falha; `TRY_CAST` devolve nulo. `lpad` recoloca zeros à esquerda.


In [4]:

sql("""
SELECT valor,
       TRY_CAST(valor AS INTEGER)                    AS como_inteiro,
       TRY_CAST(replace(valor, ',', '.') AS DOUBLE)  AS decimal_corrigido,
       lpad(valor, 5, '0')                           AS com_5_digitos
FROM (VALUES ('1151'), ('01151'), ('abc'), ('12,50')) AS t(valor)
""")


,valor,como_inteiro,decimal_corrigido,com_5_digitos
0,1151,1151,1151.0,01151
1,01151,1151,1151.0,01151
2,abc,<NA>,NaN,00abc
3,"12,50",<NA>,12.5,"12,50"



**Leitura:** `'01151'` vira o inteiro **1151**: o zero à esquerda some. `'abc'` vira nulo sem erro. `'12,50'` só vira número depois de trocar a vírgula por ponto. Um identificador com zero à esquerda (CEP, CPF, código de agência) **não é número**: é texto. Atenção: `lpad` não valida nada; `'abc'` vira `'00abc'`. Por isso, depois de corrigir, **testa-se** (exemplo A.5).



### A.3 · Texto: a mesma cidade escrita de vários jeitos
Cada função resolve uma diferença. A ordem importa: primeiro tirar acentos e espaços, depois aplicar a expressão regular.


In [5]:

sql(r"""
SELECT bruto,
       lower(trim(strip_accents(bruto)))                           AS passo_1,
       regexp_replace(lower(trim(strip_accents(bruto))), '\s+', ' ', 'g') AS passo_2,
       regexp_replace(regexp_replace(lower(trim(strip_accents(bruto))), '\s+', ' ', 'g'),
                      '\s*[/-]\s*[a-z]{2}$', '')                   AS passo_3
FROM (VALUES (' São Paulo '), ('sao  paulo'), ('SAO PAULO - SP'), ('são paulo/sp'), ('Embu-Guaçu')) AS t(bruto)
""")


,bruto,passo_1,passo_2,passo_3
0,São Paulo,sao paulo,sao paulo,sao paulo
1,sao paulo,sao paulo,sao paulo,sao paulo
2,SAO PAULO - SP,sao paulo - sp,sao paulo - sp,sao paulo
3,são paulo/sp,sao paulo/sp,sao paulo/sp,sao paulo
4,Embu-Guaçu,embu-guacu,embu-guacu,embu-guacu



**Leitura:** passo 1 tira acento, maiúsculas e espaços das pontas; passo 2 junta espaços duplos (`\s+` é "um ou mais espaços"); passo 3 remove um sufixo de UF depois de `/` ou `-` (`[a-z]{2}$` é "duas letras no fim"). As quatro primeiras viram `sao paulo`. E `embu-guacu` **continua igual**: o hífen faz parte do nome, porque a regra só remove o sufixo quando sobram exatamente duas letras depois dele.



### A.4 · Duplicatas: `DISTINCT` × uma linha por grupo
`DISTINCT` só remove linhas **idênticas**. Para ficar com uma linha por chave, escolhendo qual (a mais recente, por exemplo), usa-se `ROW_NUMBER()` com `QUALIFY`.


In [6]:

exemplo = """
SELECT * FROM (VALUES
  ('c1', 'Campinas', DATE '2018-01-10'),
  ('c1', 'Campinas', DATE '2018-01-10'),
  ('c2', 'Santos',   DATE '2017-05-02'),
  ('c2', 'Curitiba', DATE '2018-03-15')) AS t(cliente, cidade, atualizado_em)
"""
print("DISTINCT:");          display(sql(f"SELECT DISTINCT * FROM ({exemplo})"))
print("Uma linha por cliente, a mais recente:")
sql(f"""
SELECT * FROM ({exemplo})
QUALIFY ROW_NUMBER() OVER (PARTITION BY cliente ORDER BY atualizado_em DESC) = 1
""")


DISTINCT:


,cliente,cidade,atualizado_em
0,c1,Campinas,2018-01-10
1,c2,Santos,2017-05-02
2,c2,Curitiba,2018-03-15


Uma linha por cliente, a mais recente:


,cliente,cidade,atualizado_em
0,c1,Campinas,2018-01-10
1,c2,Curitiba,2018-03-15



**Leitura:** `DISTINCT` resolve `c1` (duplicata **exata**) mas mantém as duas linhas de `c2`, que diferem na cidade (duplicata **lógica**). A segunda consulta fica com uma linha por cliente, e a regra de escolha ("a mais recente") é uma **decisão** que precisa ser registrada.



### A.5 · Registrar a regra e testar o resultado
Um teste de qualidade é uma consulta que devolve as linhas que **violam** a regra. Zero linhas: passou.


In [7]:

testar("pagamento com valor negativo", "SELECT * FROM payments WHERE payment_value < 0")
testar("pagamento com zero parcelas",  "SELECT * FROM payments WHERE payment_installments = 0")

registrar("payments", "zero parcelas: marcar para revisão", "marca",
          con.sql("SELECT COUNT(*) FROM payments WHERE payment_installments = 0").fetchall()[0][0])
pd.DataFrame(REGRAS)


PASSOU | pagamento com valor negativo | 0 linha(s) violando a regra
FALHOU | pagamento com zero parcelas | 2 linha(s) violando a regra


,tabela,regra,acao,linhas_afetadas
0,payments,zero parcelas: marcar para revisão,marca,2



**Leitura:** o primeiro teste passa; o segundo falha com **2** linhas. Um teste que falha não é um defeito do notebook: é a informação de que a regra precisa de uma decisão. A tabela `REGRAS` é o **log de regras**, que acompanha a silver.



---
# Parte 1 · Medir antes de limpar
Exercícios 1 e 2. Não altere nada ainda: o objetivo é descobrir o que está errado e decidir o que fazer.


In [10]:

REGRAS.clear()   # o log começa vazio para os exercícios
def contar(q):
    return con.sql(q).fetchone()[0]

UFS = ['ac','al','ap','am','ba','ce','df','es','go','ma','mt','ms','mg','pa','pb',
       'pr','pe','pi','rj','rn','rs','ro','rr','sc','sp','se','to']
ESTADOS = ['acre','alagoas','amapa','amazonas','bahia','ceara','distrito federal',
           'espirito santo','goias','maranhao','mato grosso do sul','mato grosso',
           'minas gerais','paraiba','parana','para','pernambuco','piaui','rio de janeiro',
           'rio grande do norte','rio grande do sul','rondonia','roraima','santa catarina',
           'sao paulo','sergipe','tocantins']
UF_RE  = "|".join(UFS)
EST_RE = "|".join(ESTADOS)

def padroniza_cidade(col, uf_col=None):
    """Monta a expressão SQL que padroniza uma coluna de cidade."""
    e = f"lower(trim(strip_accents({col})))"
    e = rf"regexp_replace({e}, '\s+', ' ', 'g')"
    e = rf"regexp_replace({e}, '\s*\([^)]*\)$', '')"
    e = rf"regexp_replace({e}, '\s*[/\\,]\s*({UF_RE}|{EST_RE})$', '')"
    e = rf"regexp_replace({e}, '\s*-\s*({UF_RE})$', '')"
    e = rf"regexp_replace({e}, '(\s+-\s*|\s*-\s+)({EST_RE})$', '')"
    if uf_col:
        e = f"(CASE WHEN {e} LIKE ('% ' || lower({uf_col})) THEN left({e}, length({e}) - 3) ELSE {e} END)"
    return rf"trim(regexp_replace({e}, '\s+', ' ', 'g'))"



## 1 · O perfil das tabelas
1. Rode `SUMMARIZE` em `customers`, `sellers`, `geolocation` e `orders`.
2. Qual é o **tipo** do CEP (`..._zip_code_prefix`) em cada tabela? O que isso pode causar num JOIN?
3. Olhe o **mínimo** e o **máximo** de `seller_zip_code_prefix` e de `seller_city`. O que eles revelam?
4. Em `orders`, quais colunas têm nulos, e em que percentual?
5. Em `geolocation`, compare `approx_unique` do CEP com `count`. O que isso sugere?
6. Liste **cinco problemas** para investigar na Parte 2.


In [11]:

# Dica: sql("SELECT column_name, column_type, min, max, approx_unique, null_percentage, count FROM (SUMMARIZE sellers)")
# Repita para as outras tabelas e compare a coluna column_type dos CEPs.
for t in ["customers", "sellers", "geolocation", "orders"]:
    print(f"\n===== {t} =====")
    display(sql(f"""
        SELECT column_name, column_type, min, max, approx_unique, null_percentage, count
        FROM (SUMMARIZE {t})
    """))

pd.concat([
    sql(f"DESCRIBE {t}").query("column_name.str.contains('zip')").assign(tabela=t)
    for t in ["customers", "sellers", "geolocation"]
])[["tabela", "column_name", "column_type"]]


===== customers =====


,column_name,column_type,min,max,approx_unique,null_percentage,count
0,customer_id,VARCHAR,00012a2ce6f8dcda20d059ce98491703,ffffe8b65bbe3087b653a978c870db99,101221,0.0,99441
1,customer_unique_id,VARCHAR,0000366f3b9a7992bf8c76cfdf3221e2,ffffd2657e2aad2907e67c3e9daecbeb,93296,0.0,99441
2,customer_zip_code_prefix,BIGINT,1003,99990,14959,0.0,99441
3,customer_city,VARCHAR,abadia dos dourados,zortea,5024,0.0,99441
4,customer_state,VARCHAR,AC,TO,28,0.0,99441



===== sellers =====


,column_name,column_type,min,max,approx_unique,null_percentage,count
0,seller_id,VARCHAR,0015a82c2db000af6aaaf3ae2ecb0532,ffff564a4f9085cd26170f4732393726,3886,0.0,3095
1,seller_zip_code_prefix,BIGINT,1001,99730,2466,0.0,3095
2,seller_city,VARCHAR,4482255,xaxim,746,0.0,3095
3,seller_state,VARCHAR,AC,SP,24,0.0,3095



===== geolocation =====


,column_name,column_type,min,max,approx_unique,null_percentage,count
0,geolocation_zip_code_prefix,VARCHAR,01001,99990,16453,0.0,1000163
1,geolocation_lat,DOUBLE,-36.6053744107061,45.06593318269697,883419,0.0,1000163
2,geolocation_lng,DOUBLE,-101.46676644931476,121.10539381057764,852313,0.0,1000163
3,geolocation_city,VARCHAR,* cidade,óleo,8915,0.0,1000163
4,geolocation_state,VARCHAR,AC,TO,28,0.0,1000163



===== orders =====


,column_name,column_type,min,max,approx_unique,null_percentage,count
0,order_id,VARCHAR,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244,87618,0.00,99441
1,customer_id,VARCHAR,00012a2ce6f8dcda20d059ce98491703,ffffe8b65bbe3087b653a978c870db99,101221,0.00,99441
2,order_status,VARCHAR,approved,unavailable,9,0.00,99441
3,order_purchase_timestamp,TIMESTAMP,2016-09-04 21:15:19,2018-10-17 17:30:18,93884,0.00,99441
4,order_approved_at,TIMESTAMP,2016-09-15 12:16:38,2018-09-03 17:40:06,100122,0.16,99441
5,order_delivered_carrier_date,TIMESTAMP,2016-10-08 10:34:01,2018-09-11 19:48:28,79693,1.79,99441
6,order_delivered_customer_date,TIMESTAMP,2016-10-11 13:46:32,2018-10-17 13:22:46,89604,2.98,99441
7,order_estimated_delivery_date,TIMESTAMP,2016-09-30 00:00:00,2018-11-12 00:00:00,440,0.00,99441


,tabela,column_name,column_type
2,customers,customer_zip_code_prefix,BIGINT
1,sellers,seller_zip_code_prefix,BIGINT
0,geolocation,geolocation_zip_code_prefix,VARCHAR



## 2 · Nem todo nulo é erro
1. Conte, **por status do pedido**, quantos pedidos não têm data de aprovação, de coleta pela transportadora e de entrega ao cliente.
2. Para cada combinação, decida: o nulo é **esperado** (o evento ainda não aconteceu) ou é uma **falha** (o evento aconteceu e a data não foi gravada)?
3. Quantos pedidos `delivered` (entregues) não têm data de entrega? E quantos `canceled` (cancelados) **têm** data de entrega?
4. Registre com `registrar()` a regra que você adotaria para cada caso. Não imputar (inventar) datas é uma regra válida.


In [12]:

# Dica: SELECT order_status, COUNT(*) AS pedidos,
#               COUNT(*) - COUNT(order_approved_at) AS sem_aprovacao, ...
#        FROM orders GROUP BY 1 ORDER BY 2 DESC
# COUNT(coluna) conta só os valores não nulos.

sql("""
SELECT order_status,
       COUNT(*)                                        AS pedidos,
       COUNT(*) - COUNT(order_approved_at)             AS sem_aprovacao,
       COUNT(*) - COUNT(order_delivered_carrier_date)  AS sem_coleta,
       COUNT(*) - COUNT(order_delivered_customer_date) AS sem_entrega
FROM orders
GROUP BY 1
ORDER BY 2 DESC
""")
entregue_sem_data  = contar("SELECT COUNT(*) FROM orders WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL")
entregue_sem_aprov = contar("SELECT COUNT(*) FROM orders WHERE order_status = 'delivered' AND order_approved_at IS NULL")
cancelado_com_data = contar("SELECT COUNT(*) FROM orders WHERE order_status = 'canceled' AND order_delivered_customer_date IS NOT NULL")
print("Entregues sem data de entrega:", entregue_sem_data)
print("Entregues sem data de aprovação:", entregue_sem_aprov)
print("Cancelados COM data de entrega:", cancelado_com_data)

registrar("orders", "nulo em datas de pedidos não entregues (created, approved, shipped, invoiced...): esperado, mantém nulo", "corrige", 0)
registrar("orders", "delivered sem data de entrega: não imputar, marcar flag_entregue_sem_data", "marca", entregue_sem_data)
registrar("orders", "delivered sem data de aprovação: não imputar, manter nulo", "marca", entregue_sem_aprov)
registrar("orders", "canceled com data de entrega: marcar flag_cancelado_com_data (decisão do negócio)", "marca", cancelado_com_data)
pd.DataFrame(REGRAS)


Entregues sem data de entrega: 8
Entregues sem data de aprovação: 14
Cancelados COM data de entrega: 6


,tabela,regra,acao,linhas_afetadas
0,orders,nulo em datas de pedidos não entregues (create...,corrige,0
1,orders,"delivered sem data de entrega: não imputar, ma...",marca,8
2,orders,"delivered sem data de aprovação: não imputar, ...",marca,14
3,orders,canceled com data de entrega: marcar flag_canc...,marca,6



---
# Parte 2 · Corrigir, padronizar, deduplicar e montar a silver
Exercícios 3 a 7. **Prioridade:** 3, 4, 5 e 7. O 6 pode ir para o debate se faltar tempo.



## 3 · O CEP que perdeu o zero
1. Quantos clientes e quantos vendedores têm CEP com **menos de 5 dígitos** (valor menor que 10000)?
2. Faça o JOIN de `customers` com os CEPs distintos de `geolocation` de dois jeitos: (a) convertendo o CEP do cliente para texto com `CAST`; (b) com `lpad(..., 5, '0')`. Quantos clientes encontram correspondência em cada caso?
3. Depois da correção, quantos clientes continuam **sem** nenhuma geolocalização?
4. Registre a regra.


In [15]:

# Dica: WITH g AS (SELECT DISTINCT geolocation_zip_code_prefix AS cep FROM geolocation)
#       SELECT COUNT(*) FROM customers c JOIN g ON CAST(c.customer_zip_code_prefix AS VARCHAR) = g.cep
# Depois troque o CAST por lpad(CAST(c.customer_zip_code_prefix AS VARCHAR), 5, '0')

display(sql("""
SELECT
  (SELECT COUNT(*) FROM customers WHERE TRY_CAST(customer_zip_code_prefix AS INTEGER) < 10000) AS clientes_cep_curto,
  (SELECT COUNT(*) FROM sellers   WHERE TRY_CAST(seller_zip_code_prefix   AS INTEGER) < 10000) AS vendedores_cep_curto
"""))
sql("""
WITH g AS (
  SELECT DISTINCT lpad(CAST(geolocation_zip_code_prefix AS VARCHAR), 5, '0') AS cep
  FROM geolocation
)
SELECT
  (SELECT COUNT(*) FROM customers) AS total_clientes,
  (SELECT COUNT(*) FROM customers c
    WHERE CAST(c.customer_zip_code_prefix AS VARCHAR) IN (SELECT cep FROM g))                AS match_com_cast,
  (SELECT COUNT(*) FROM customers c
    WHERE lpad(CAST(c.customer_zip_code_prefix AS VARCHAR), 5, '0') IN (SELECT cep FROM g))  AS match_com_lpad
""")

,clientes_cep_curto,vendedores_cep_curto
0,23995,1027


,total_clientes,match_com_cast,match_com_lpad
0,99441,75176,99162



## 4 · A mesma cidade de muitos jeitos
1. Em `geolocation`, quantas cidades distintas existem antes e depois de aplicar `lower(trim(strip_accents(...)))`?
2. Em `sellers`, aplique a regra completa: minúsculas, sem acento, espaços simples, sem texto entre parênteses no fim, sem sufixo de UF ou estado depois de `/`, `\` ou `-`, e sem a sigla da própria UF no fim. Quantas cidades distintas antes e depois? Quantas linhas mudaram?
3. Crie uma **tabela de referência** de cidades e UFs a partir de `geolocation` (padronizada) e verifique quantos vendedores têm a cidade: (a) encontrada na mesma UF; (b) encontrada só em outra UF; (c) não encontrada.
4. *(Opcional)* Para as não encontradas, sugira a cidade mais parecida da mesma UF com `jaro_winkler_similarity`. A partir de que semelhança a sugestão parece confiável?


In [16]:

# Dica: crie a regra como texto e reaproveite:
# REGRA = r"""regexp_replace(lower(trim(strip_accents(seller_city))), '\s+', ' ', 'g')"""
# e vá envolvendo com outros regexp_replace, um problema de cada vez.
# Referência: CREATE TABLE ref_cidades AS SELECT DISTINCT lower(trim(strip_accents(geolocation_city))) AS cidade,
#             geolocation_state AS uf FROM geolocation

sql("""
SELECT COUNT(DISTINCT geolocation_city)                             AS cidades_antes,
       COUNT(DISTINCT lower(trim(strip_accents(geolocation_city)))) AS cidades_depois
FROM geolocation
""")

,cidades_antes,cidades_depois
0,8011,5968


In [17]:
PC_SELLER = padroniza_cidade("seller_city", "seller_state")

display(sql(f"""
SELECT COUNT(DISTINCT seller_city) AS cidades_antes,
       COUNT(DISTINCT {PC_SELLER}) AS cidades_depois,
       COUNT(*) FILTER (WHERE seller_city IS DISTINCT FROM {PC_SELLER}) AS linhas_que_mudaram
FROM sellers
"""))

# O que mudou
sql(f"""
SELECT seller_city AS antes, {PC_SELLER} AS depois, seller_state, COUNT(*) AS vendedores
FROM sellers
WHERE seller_city IS DISTINCT FROM {PC_SELLER}
GROUP BY ALL
ORDER BY vendedores DESC
""")

,cidades_antes,cidades_depois,linhas_que_mudaram
0,611,587,29


,antes,depois,seller_state,vendedores
0,sao paulo - sp,sao paulo,SP,3
1,ferraz de vasconcelos,ferraz de vasconcelos,SP,1
2,sao paulo / sao paulo,sao paulo,SP,1
3,são paulo,sao paulo,SP,1
4,cariacica / es,cariacica,ES,1
5,sbc/sp,sbc,SP,1
6,santo andre/sao paulo,santo andre,SP,1
7,maua/sao paulo,maua,SP,1
8,mogi das cruzes / sp,mogi das cruzes,SP,1
9,rio de janeiro \rio de janeiro,rio de janeiro,RJ,1


In [18]:
con.execute(f"""
CREATE OR REPLACE TABLE ref_cidades AS
SELECT DISTINCT {padroniza_cidade('geolocation_city')} AS cidade,
       upper(geolocation_state) AS uf
FROM geolocation
""")

con.execute(f"""
CREATE OR REPLACE TABLE sellers_cidade AS
WITH s AS (
  SELECT seller_id, seller_city, upper(seller_state) AS uf, {PC_SELLER} AS cidade_pad
  FROM sellers
)
SELECT s.*,
       CASE
         WHEN EXISTS (SELECT 1 FROM ref_cidades r WHERE r.cidade = s.cidade_pad AND r.uf = s.uf) THEN 'mesma_uf'
         WHEN EXISTS (SELECT 1 FROM ref_cidades r WHERE r.cidade = s.cidade_pad)                 THEN 'outra_uf'
         ELSE 'nao_encontrada'
       END AS flag_cidade
FROM s
""")

display(sql("SELECT flag_cidade, COUNT(*) AS vendedores FROM sellers_cidade GROUP BY 1 ORDER BY 2 DESC"))

# Quem caiu em 'outra_uf' e 'nao_encontrada'
sql("""
SELECT flag_cidade, seller_city, cidade_pad, uf, COUNT(*) AS vendedores
FROM sellers_cidade
WHERE flag_cidade <> 'mesma_uf'
GROUP BY ALL
ORDER BY flag_cidade, vendedores DESC
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,flag_cidade,vendedores
0,mesma_uf,3021
1,nao_encontrada,39
2,outra_uf,35


,flag_cidade,seller_city,cidade_pad,uf,vendedores
0,nao_encontrada,santa barbara d´oeste,santa barbara d´oeste,SP,2
1,nao_encontrada,vendas@creditparts.com.br,vendas@creditparts.com.br,PR,1
2,nao_encontrada,scao jose do rio pardo,scao jose do rio pardo,SP,1
3,nao_encontrada,aguas claras df,aguas claras df,SP,1
4,nao_encontrada,robeirao preto,robeirao preto,SP,1
...,...,...,...,...,...
62,outra_uf,vila velha,vila velha,SP,1
63,outra_uf,juiz de fora,juiz de fora,SP,1
64,outra_uf,porto alegre,porto alegre,SP,1
65,outra_uf,laguna,laguna,SP,1


In [19]:
sql("""
SELECT s.seller_city, s.cidade_pad, s.uf, r.cidade AS sugestao,
       round(jaro_winkler_similarity(s.cidade_pad, r.cidade), 3) AS similaridade
FROM sellers_cidade s
JOIN ref_cidades r ON r.uf = s.uf
WHERE s.flag_cidade = 'nao_encontrada'
QUALIFY ROW_NUMBER() OVER (PARTITION BY s.seller_id
                           ORDER BY jaro_winkler_similarity(s.cidade_pad, r.cidade) DESC) = 1
ORDER BY similaridade DESC
""")

,seller_city,cidade_pad,uf,sugestao,similaridade
0,sao bernardo do capo,sao bernardo do capo,SP,sao bernardo do campo,0.990
1,sao jose do rio pret,sao jose do rio pret,SP,sao jose do rio preto,0.990
2,sao jose dos pinhas,sao jose dos pinhas,PR,sao jose dos pinhais,0.990
3,balenario camboriu,balenario camboriu,SC,balneario camboriu,0.987
4,portoferreira,portoferreira,SP,porto ferreira,0.986
5,riberao preto,riberao preto,SP,ribeirao preto,0.986
6,scao jose do rio pardo,scao jose do rio pardo,SP,sao jose do rio pardo,0.986
7,belo horizont,belo horizont,MG,belo horizonte,0.986
8,floranopolis,floranopolis,SC,florianopolis,0.985
9,juzeiro do norte,juzeiro do norte,CE,juazeiro do norte,0.984


In [21]:
# Ajusta a linha 0 (nada é corrigido: o nulo é mantido)
REGRAS[0] = {"tabela": "orders",
             "regra": "nulo em datas de etapas que ainda não ocorreram: esperado, mantém nulo",
             "acao": "marca", "linhas_afetadas": 0}

# Insere as regras do exercício 3 depois das do exercício 2
ex3 = [
    {"tabela": "customers", "regra": "CEP como texto de 5 dígitos com lpad (zero à esquerda)", "acao": "corrige", "linhas_afetadas": 23995},
    {"tabela": "sellers",   "regra": "CEP como texto de 5 dígitos com lpad (zero à esquerda)", "acao": "corrige", "linhas_afetadas": 1027},
    {"tabela": "customers", "regra": "CEP inexistente na geolocation: manter cliente, sem coordenada", "acao": "marca", "linhas_afetadas": 279},
]
REGRAS[4:4] = ex3
pd.DataFrame(REGRAS)

,tabela,regra,acao,linhas_afetadas
0,orders,nulo em datas de etapas que ainda não ocorrera...,marca,0
1,orders,"delivered sem data de entrega: não imputar, ma...",marca,8
2,orders,"delivered sem data de aprovação: não imputar, ...",marca,14
3,orders,canceled com data de entrega: marcar flag_canc...,marca,6
4,customers,CEP como texto de 5 dígitos com lpad (zero à e...,corrige,23995
5,sellers,CEP como texto de 5 dígitos com lpad (zero à e...,corrige,1027
6,customers,CEP inexistente na geolocation: manter cliente...,marca,279
7,sellers,"cidade padronizada (minúsculas, sem acento, es...",corrige,29
8,sellers,cidade encontrada só em outra UF: flag_cidade ...,marca,35
9,sellers,cidade não encontrada na referência: flag_cida...,marca,39



## 5 · Duplicata exata e duplicata lógica
1. Em `geolocation`, quantas linhas existem, quantas são **distintas** e quantos CEPs distintos?
2. Em `reviews` (avaliações), quantos `review_id` aparecem em mais de uma linha? Em quantas linhas no total?
3. Para esses `review_id` repetidos, o conteúdo (nota, título, comentário, datas) muda entre as linhas? O que isso indica?
4. Qual seria a chave correta da silver de avaliações? Teste a unicidade dela com `testar()`.


In [22]:

# Dica: SELECT COUNT(*) FROM (SELECT DISTINCT * FROM geolocation)
# Para o item 3: COUNT(DISTINCT (review_score, review_comment_title, review_comment_message,
#                               review_creation_date, review_answer_timestamp))

# Item 1: geolocation
display(sql("""
SELECT COUNT(*)                                                    AS linhas,
       (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM geolocation)) AS linhas_distintas,
       COUNT(DISTINCT geolocation_zip_code_prefix)                 AS ceps_distintos
FROM geolocation
"""))

# Item 2: review_id repetido
sql("""
WITH d AS (SELECT review_id, COUNT(*) AS n FROM reviews GROUP BY 1 HAVING COUNT(*) > 1)
SELECT COUNT(*) AS review_ids_repetidos, SUM(n) AS linhas_envolvidas FROM d
""")


,linhas,linhas_distintas,ceps_distintos
0,1000163,738332,19015


,review_ids_repetidos,linhas_envolvidas
0,789,1603.0


In [23]:
# Item 3: o conteúdo muda entre as linhas repetidas?
rep = sql("""
SELECT review_id,
       COUNT(*)                 AS linhas,
       COUNT(DISTINCT order_id) AS pedidos_distintos,
       COUNT(DISTINCT (review_score, review_comment_title, review_comment_message,
                       review_creation_date, review_answer_timestamp)) AS versoes_do_conteudo
FROM reviews
GROUP BY 1
HAVING COUNT(*) > 1
""")
display(rep.head(10))
display(rep.groupby(["versoes_do_conteudo", "pedidos_distintos"]).size().rename("review_ids").reset_index())

,review_id,linhas,pedidos_distintos,versoes_do_conteudo
0,7eeb2cbff212e6c311994d274bc1a149,2,2,1
1,f3544b47fdfb4bdd2092131d3c403ae9,2,2,1
2,0c76e7a547a531e7bf9f0b99cba071c1,3,3,1
3,4219a80ab469e3fc9901437b73da3f75,3,3,1
4,7c7d0a7bb5f4b7e668c4e00a931ccbf4,2,2,1
5,a71726d518ede586d51efacfeb2ade2c,2,2,1
6,3f8bb859f2d7024bc419b489a3fcfd56,2,2,1
7,c56a88a404315a0d9e412c1472dda2c4,2,2,1
8,a3a174dbf8bf3e1ad821890ede142c99,2,2,1
9,875d4ac395bbe012ffeb7259b08aaa00,2,2,1


,versoes_do_conteudo,pedidos_distintos,review_ids
0,1,2,764
1,1,3,25


In [24]:
# Item 4: chave da silver de avaliações
testar("review_id único sozinho",     "SELECT review_id FROM reviews GROUP BY 1 HAVING COUNT(*) > 1")
testar("(review_id, order_id) único", "SELECT review_id, order_id FROM reviews GROUP BY ALL HAVING COUNT(*) > 1")

FALHOU | review_id único sozinho | 789 linha(s) violando a regra
PASSOU | (review_id, order_id) único | 0 linha(s) violando a regra


0

In [25]:
display(sql("""
SELECT COUNT(*) AS pedidos_com_mais_de_uma_avaliacao
FROM (SELECT order_id FROM reviews GROUP BY 1 HAVING COUNT(*) > 1)
"""))

,pedidos_com_mais_de_uma_avaliacao
0,547


In [26]:
sql("""
SELECT COUNT(*)                                          AS pedidos,
       COUNT(*) FILTER (WHERE notas_distintas > 1)       AS com_notas_diferentes,
       COUNT(*) FILTER (WHERE notas_distintas = 1)       AS com_mesma_nota
FROM (
  SELECT order_id, COUNT(DISTINCT review_score) AS notas_distintas
  FROM reviews GROUP BY 1 HAVING COUNT(*) > 1
)
""")

,pedidos,com_notas_diferentes,com_mesma_nota
0,547,202,345


In [28]:
ex5 = [
    {"tabela": "geolocation", "regra": "remover duplicatas exatas", "acao": "remove", "linhas_afetadas": 1000163 - 738332},
    {"tabela": "reviews", "regra": "chave = (review_id, order_id); review_id repetido é a mesma avaliação em vários pedidos, nenhuma linha removida", "acao": "corrige", "linhas_afetadas": 0},
]
REGRAS[11:11] = ex5
REGRAS[-1]["regra"] = "pedido com mais de uma avaliação (N:N): 202 com notas diferentes; manter todas na silver, escolha fica para o negócio"
pd.DataFrame(REGRAS)

,tabela,regra,acao,linhas_afetadas
0,orders,nulo em datas de etapas que ainda não ocorrera...,marca,0
1,orders,"delivered sem data de entrega: não imputar, ma...",marca,8
2,orders,"delivered sem data de aprovação: não imputar, ...",marca,14
3,orders,canceled com data de entrega: marcar flag_canc...,marca,6
4,customers,CEP como texto de 5 dígitos com lpad (zero à e...,corrige,23995
5,sellers,CEP como texto de 5 dígitos com lpad (zero à e...,corrige,1027
6,customers,CEP inexistente na geolocation: manter cliente...,marca,279
7,sellers,"cidade padronizada (minúsculas, sem acento, es...",corrige,29
8,sellers,cidade encontrada só em outra UF: flag_cidade ...,marca,35
9,sellers,cidade não encontrada na referência: flag_cida...,marca,39



## 6 · Fora do padrão não é o mesmo que errado
1. **Datas incoerentes em `orders`:** quantos pedidos foram coletados pela transportadora antes da compra? Antes da aprovação? Quantos foram entregues ao cliente antes da coleta?
2. **Produtos:** quantos têm peso igual a 0? De que categoria?
3. **Itens:** quantos têm frete igual a 0? Em quantos pedidos?
4. **Preço:** calcule Q1, Q3, IQR e o limite Q3 + 1,5 × IQR. Quantos itens ficam acima? Repita o cálculo com o logaritmo do preço (`ln(price)`) e compare. Compare também média e mediana.
5. **Geolocalização:** quantas linhas e quantos CEPs têm coordenadas fora do retângulo que contém o Brasil (latitude entre −33,8 e 5,3; longitude entre −74,0 e −34,7)?
6. Para cada caso, decida: corrigir, marcar ou remover?


In [29]:

# Dica: COUNT(*) FILTER (WHERE order_delivered_carrier_date < order_purchase_timestamp)
# Quartis: quantile_cont(price, 0.25) e quantile_cont(price, 0.75)
# Item 1: datas incoerentes
sql("""
SELECT
  COUNT(*) FILTER (WHERE order_delivered_carrier_date  < order_purchase_timestamp)     AS coleta_antes_compra,
  COUNT(*) FILTER (WHERE order_delivered_carrier_date  < order_approved_at)            AS coleta_antes_aprovacao,
  COUNT(*) FILTER (WHERE order_delivered_customer_date < order_delivered_carrier_date) AS entrega_antes_coleta
FROM orders
""")


,coleta_antes_compra,coleta_antes_aprovacao,entrega_antes_coleta
0,166,1359,23


In [30]:
# Item 2: produtos com peso 0
sql("""
SELECT product_category_name, COUNT(*) AS produtos
FROM products WHERE product_weight_g = 0
GROUP BY 1 ORDER BY 2 DESC
""")

,product_category_name,produtos
0,cama_mesa_banho,4


In [31]:
# Item 3: frete zero
sql("""
SELECT COUNT(*) AS itens_frete_zero, COUNT(DISTINCT order_id) AS pedidos
FROM items WHERE freight_value = 0
""")

,itens_frete_zero,pedidos
0,383,339


In [32]:
# Item 4: IQR no preço e no log do preço
sql("""
WITH q AS (
  SELECT quantile_cont(price, 0.25)     AS q1,  quantile_cont(price, 0.75)     AS q3,
         quantile_cont(ln(price), 0.25) AS lq1, quantile_cont(ln(price), 0.75) AS lq3,
         avg(price) AS media, median(price) AS mediana
  FROM items
)
SELECT q1, q3, q3 - q1 AS iqr,
       q3 + 1.5 * (q3 - q1)                                                   AS limite,
       (SELECT COUNT(*) FROM items WHERE price > q3 + 1.5 * (q3 - q1))        AS acima_limite,
       exp(lq3 + 1.5 * (lq3 - lq1))                                           AS limite_log_em_reais,
       (SELECT COUNT(*) FROM items WHERE ln(price) > lq3 + 1.5 * (lq3 - lq1)) AS acima_limite_log,
       (SELECT COUNT(*) FROM items)                                           AS total_itens,
       media, mediana
FROM q
""")

,q1,q3,iqr,limite,acima_limite,limite_log_em_reais,acima_limite_log,total_itens,media,mediana
0,39.9,134.9,95.0,277.4,8427,838.630263,1282,112650,120.653739,74.99


In [33]:
# Item 5: coordenadas fora do Brasil
FORA_BR = "NOT (geolocation_lat BETWEEN -33.8 AND 5.3 AND geolocation_lng BETWEEN -74.0 AND -34.7)"
sql(f"""
SELECT COUNT(*) FILTER (WHERE {FORA_BR})                                   AS linhas_fora,
       COUNT(DISTINCT geolocation_zip_code_prefix) FILTER (WHERE {FORA_BR}) AS ceps_fora
FROM geolocation
""")


,linhas_fora,ceps_fora
0,42,21


In [34]:
registrar("orders", "datas incoerentes (coleta antes da compra/aprovação, entrega antes da coleta): flag, sem corrigir", "marca",
          contar("""SELECT COUNT(*) FROM orders
                    WHERE order_delivered_carrier_date  < order_purchase_timestamp
                       OR order_delivered_carrier_date  < order_approved_at
                       OR order_delivered_customer_date < order_delivered_carrier_date"""))
registrar("products", "peso 0 vira nulo (peso desconhecido, não zero)", "corrige", 4)
registrar("items", "frete 0: flag_frete_zero (pode ser frete grátis legítimo)", "marca", 383)
registrar("items", "preço acima do limite IQR no log (R$ 838,63): não remover, cauda longa é real", "marca", 1282)
registrar("geolocation", "coordenada fora do retângulo do Brasil: flag_fora_do_brasil", "marca",
          contar(f"SELECT COUNT(*) FROM (SELECT DISTINCT * FROM geolocation) WHERE {FORA_BR}"))
pd.DataFrame(REGRAS)

,tabela,regra,acao,linhas_afetadas
0,orders,nulo em datas de etapas que ainda não ocorrera...,marca,0
1,orders,"delivered sem data de entrega: não imputar, ma...",marca,8
2,orders,"delivered sem data de aprovação: não imputar, ...",marca,14
3,orders,canceled com data de entrega: marcar flag_canc...,marca,6
4,customers,CEP como texto de 5 dígitos com lpad (zero à e...,corrige,23995
5,sellers,CEP como texto de 5 dígitos com lpad (zero à e...,corrige,1027
6,customers,CEP inexistente na geolocation: manter cliente...,marca,279
7,sellers,"cidade padronizada (minúsculas, sem acento, es...",corrige,29
8,sellers,cidade encontrada só em outra UF: flag_cidade ...,marca,35
9,sellers,cidade não encontrada na referência: flag_cida...,marca,39



## 7 · Montar a silver
Construa as tabelas da camada silver, aplicando as regras dos exercícios 2 a 6. **Nenhuma linha válida pode sumir.**
1. `silver_customers` e `silver_sellers`: CEP como texto de 5 dígitos; em vendedores, cidade padronizada e a flag `flag_cidade` com a situação do exercício 4.
2. `silver_geolocation`: sem duplicatas exatas, com a cidade padronizada e a flag `flag_fora_do_brasil`.
3. `silver_orders`: todas as linhas, com flags para entregue sem data, cancelado com data e datas incoerentes.
4. `silver_products`: peso 0 vira nulo. `silver_items`: flag de frete zero.
5. Rode os testes e compare as linhas de cada tabela no bronze e na silver.


In [35]:

# Dica: CREATE OR REPLACE TABLE silver_customers AS
#       SELECT customer_id, customer_unique_id,
#              lpad(CAST(customer_zip_code_prefix AS VARCHAR), 5, '0') AS cep_prefixo, ...
#       FROM customers
# Flags: (condição) AS flag_nome   ->  vira verdadeiro ou falso
PC_CUST = padroniza_cidade("customer_city", "customer_state")
PC_GEO  = padroniza_cidade("geolocation_city")

con.execute(f"""
CREATE OR REPLACE TABLE silver_customers AS
SELECT customer_id, customer_unique_id,
       lpad(CAST(customer_zip_code_prefix AS VARCHAR), 5, '0') AS cep_prefixo,
       {PC_CUST}             AS cidade,
       upper(customer_state) AS uf
FROM customers
""")

con.execute("""
CREATE OR REPLACE TABLE silver_sellers AS
SELECT s.seller_id,
       lpad(CAST(s.seller_zip_code_prefix AS VARCHAR), 5, '0') AS cep_prefixo,
       c.cidade_pad AS cidade,
       c.uf,
       c.flag_cidade
FROM sellers s
JOIN sellers_cidade c USING (seller_id)
""")

con.execute(f"""
CREATE OR REPLACE TABLE silver_geolocation AS
SELECT DISTINCT
       lpad(CAST(geolocation_zip_code_prefix AS VARCHAR), 5, '0') AS cep_prefixo,
       geolocation_lat AS lat,
       geolocation_lng AS lng,
       {PC_GEO}                 AS cidade,
       upper(geolocation_state) AS uf,
       {FORA_BR}                AS flag_fora_do_brasil
FROM (SELECT DISTINCT * FROM geolocation)
""")

con.execute("""
CREATE OR REPLACE TABLE silver_orders AS
SELECT *,
       (order_status = 'delivered' AND order_delivered_customer_date IS NULL)     AS flag_entregue_sem_data,
       (order_status = 'canceled'  AND order_delivered_customer_date IS NOT NULL) AS flag_cancelado_com_data,
       coalesce(order_delivered_carrier_date  < order_purchase_timestamp, false)
    OR coalesce(order_delivered_carrier_date  < order_approved_at, false)
    OR coalesce(order_delivered_customer_date < order_delivered_carrier_date, false) AS flag_datas_incoerentes
FROM orders
""")

con.execute("""
CREATE OR REPLACE TABLE silver_products AS
SELECT * REPLACE (CASE WHEN product_weight_g = 0 THEN NULL ELSE product_weight_g END AS product_weight_g)
FROM products
""")

con.execute("""
CREATE OR REPLACE TABLE silver_items AS
SELECT *, (freight_value = 0) AS flag_frete_zero
FROM items
""")

con.execute("""
CREATE OR REPLACE TABLE silver_reviews AS
SELECT * FROM reviews
""")
print("Silver criada.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Silver criada.


In [36]:
testar("CEP de clientes com 5 dígitos",   "SELECT * FROM silver_customers WHERE NOT regexp_full_match(cep_prefixo, '[0-9]{5}')")
testar("CEP de vendedores com 5 dígitos", "SELECT * FROM silver_sellers   WHERE NOT regexp_full_match(cep_prefixo, '[0-9]{5}')")
testar("customer_id único",  "SELECT customer_id FROM silver_customers GROUP BY 1 HAVING COUNT(*) > 1")
testar("seller_id único",    "SELECT seller_id   FROM silver_sellers   GROUP BY 1 HAVING COUNT(*) > 1")
testar("order_id único",     "SELECT order_id    FROM silver_orders    GROUP BY 1 HAVING COUNT(*) > 1")
testar("(review_id, order_id) único", "SELECT review_id, order_id FROM silver_reviews GROUP BY ALL HAVING COUNT(*) > 1")
testar("geolocation sem duplicata exata", "SELECT * FROM silver_geolocation GROUP BY ALL HAVING COUNT(*) > 1")
testar("nenhum peso 0 em produtos",       "SELECT * FROM silver_products WHERE product_weight_g = 0")
testar("flags de orders sem nulo", """SELECT * FROM silver_orders WHERE flag_entregue_sem_data IS NULL
                                      OR flag_cancelado_com_data IS NULL OR flag_datas_incoerentes IS NULL""")
testar("flag_datas_incoerentes = 1382", "SELECT 1 WHERE (SELECT COUNT(*) FROM silver_orders WHERE flag_datas_incoerentes) <> 1382")
testar("nenhum cliente sumiu",  "SELECT 1 WHERE (SELECT COUNT(*) FROM customers) <> (SELECT COUNT(*) FROM silver_customers)")
testar("nenhum vendedor sumiu", "SELECT 1 WHERE (SELECT COUNT(*) FROM sellers)   <> (SELECT COUNT(*) FROM silver_sellers)")
testar("nenhum pedido sumiu",   "SELECT 1 WHERE (SELECT COUNT(*) FROM orders)    <> (SELECT COUNT(*) FROM silver_orders)")
testar("nenhum item sumiu",     "SELECT 1 WHERE (SELECT COUNT(*) FROM items)     <> (SELECT COUNT(*) FROM silver_items)");

PASSOU | CEP de clientes com 5 dígitos | 0 linha(s) violando a regra
PASSOU | CEP de vendedores com 5 dígitos | 0 linha(s) violando a regra
PASSOU | customer_id único | 0 linha(s) violando a regra
PASSOU | seller_id único | 0 linha(s) violando a regra
PASSOU | order_id único | 0 linha(s) violando a regra
PASSOU | (review_id, order_id) único | 0 linha(s) violando a regra
PASSOU | geolocation sem duplicata exata | 0 linha(s) violando a regra
PASSOU | nenhum peso 0 em produtos | 0 linha(s) violando a regra
PASSOU | flags de orders sem nulo | 0 linha(s) violando a regra
PASSOU | flag_datas_incoerentes = 1382 | 0 linha(s) violando a regra
PASSOU | nenhum cliente sumiu | 0 linha(s) violando a regra
PASSOU | nenhum vendedor sumiu | 0 linha(s) violando a regra
PASSOU | nenhum pedido sumiu | 0 linha(s) violando a regra
PASSOU | nenhum item sumiu | 0 linha(s) violando a regra


In [37]:
pares = {"customers": "silver_customers", "sellers": "silver_sellers", "geolocation": "silver_geolocation",
         "orders": "silver_orders", "products": "silver_products", "items": "silver_items", "reviews": "silver_reviews"}
display(pd.DataFrame([{"tabela": b,
                       "bronze": contar(f"SELECT COUNT(*) FROM {b}"),
                       "silver": contar(f"SELECT COUNT(*) FROM {s}")} for b, s in pares.items()])
        .assign(diferenca=lambda d: d.silver - d.bronze))

pd.DataFrame(REGRAS)

,tabela,bronze,silver,diferenca
0,customers,99441,99441,0
1,sellers,3095,3095,0
2,geolocation,1000163,720496,-279667
3,orders,99441,99441,0
4,products,32951,32951,0
5,items,112650,112650,0
6,reviews,99224,99224,0


,tabela,regra,acao,linhas_afetadas
0,orders,nulo em datas de etapas que ainda não ocorrera...,marca,0
1,orders,"delivered sem data de entrega: não imputar, ma...",marca,8
2,orders,"delivered sem data de aprovação: não imputar, ...",marca,14
3,orders,canceled com data de entrega: marcar flag_canc...,marca,6
4,customers,CEP como texto de 5 dígitos com lpad (zero à e...,corrige,23995
5,sellers,CEP como texto de 5 dígitos com lpad (zero à e...,corrige,1027
6,customers,CEP inexistente na geolocation: manter cliente...,marca,279
7,sellers,"cidade padronizada (minúsculas, sem acento, es...",corrige,29
8,sellers,cidade encontrada só em outra UF: flag_cidade ...,marca,35
9,sellers,cidade não encontrada na referência: flag_cida...,marca,39


In [38]:
registrar("geolocation", "linhas que só diferiam na grafia da cidade viraram duplicatas após padronizar: removidas", "remove",
          contar("SELECT COUNT(*) FROM (SELECT DISTINCT * FROM geolocation)") - contar("SELECT COUNT(*) FROM silver_geolocation"))
pd.DataFrame(REGRAS)

,tabela,regra,acao,linhas_afetadas
0,orders,nulo em datas de etapas que ainda não ocorrera...,marca,0
1,orders,"delivered sem data de entrega: não imputar, ma...",marca,8
2,orders,"delivered sem data de aprovação: não imputar, ...",marca,14
3,orders,canceled com data de entrega: marcar flag_canc...,marca,6
4,customers,CEP como texto de 5 dígitos com lpad (zero à e...,corrige,23995
5,sellers,CEP como texto de 5 dígitos com lpad (zero à e...,corrige,1027
6,customers,CEP inexistente na geolocation: manter cliente...,marca,279
7,sellers,"cidade padronizada (minúsculas, sem acento, es...",corrige,29
8,sellers,cidade encontrada só em outra UF: flag_cidade ...,marca,35
9,sellers,cidade não encontrada na referência: flag_cida...,marca,39


In [39]:
pd.DataFrame(REGRAS).to_csv("log_regras_silver.csv", index=False)
for t in ["silver_customers", "silver_sellers", "silver_geolocation", "silver_orders",
          "silver_products", "silver_items", "silver_reviews"]:
    con.execute(f"COPY {t} TO '{t}.parquet' (FORMAT PARQUET)")
print("Arquivos salvos.")

Arquivos salvos.


---
## Registro da equipe
Preencha antes do debate.

| Pergunta | Resposta da equipe |
|---|---|
| Os três problemas de qualidade mais graves que encontramos | **1. CEP lido como número:** 23.995 clientes (24%) e 1.027 vendedores (33%) perderam o zero à esquerda. Com `CAST`, só 75,6% dos clientes encontravam geolocalização; com `lpad`, 99,7%. A Grande SP sumia do JOIN sem nenhum erro.<br>**2. Relação N:N entre pedidos e avaliações:** 789 `review_id` cobrem vários pedidos e 547 pedidos têm várias avaliações (202 com notas diferentes). Um JOIN direto infla contagens e médias.<br>**3. `geolocation` redundante:** 279.667 linhas removidas (28%): 261.831 duplicatas exatas e 17.836 que só diferiam na grafia da cidade. Mesmo depois, há ~38 coordenadas por CEP, o que exige agregação antes de qualquer JOIN. |
| Uma regra que decidimos **não** aplicar, e por quê | **Remover preços extremos pelo IQR.** A distribuição tem cauda longa (média R$ 120,65 × mediana R$ 74,99). O IQR no preço bruto marcaria 8.427 itens (7,5%), e mesmo no log os 1.282 acima de R$ 838,63 são produtos caros reais. Removê-los apagaria vendas legítimas. Também não corrigimos cidades automaticamente pelo Jaro-Winkler: acima de 0,93 a sugestão serve para revisão humana (24 dos 39 casos), não para correção. |
| Um teste que falhou e o que ele revelou | **Unicidade de `review_id` falhou com 789 violações.** Não eram duplicatas: o conteúdo (nota, comentário, datas) era idêntico e só o pedido mudava. A mesma avaliação está ligada a vários pedidos. A chave correta é `(review_id, order_id)`, que passou no teste. Deduplicar por `review_id` tiraria a avaliação de pedidos reais. |
| Uma decisão que precisa de alguém do negócio | **Qual avaliação vale para o pedido** quando há mais de uma com notas diferentes (202 pedidos): a primeira, a última ou a média? A silver mantém todas, e a escolha fica para a gold.<br>Também: os **1.359 pedidos coletados antes da aprovação**. A aprovação é gravada com atraso ou a "coleta" registra a emissão da etiqueta? Por ora, apenas marcados com `flag_datas_incoerentes`. |